In [1]:
import os
import config
import warnings
import pandas as pd

In [2]:
%config InlineBackend.figure_format = 'retina'
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')
os.chdir(config.DIR_ROOT)

# Обучение модели CNN Classifier на 50 самых важных k-mer

In [3]:
# Загрузка данных
embeddings_7mer_path = os.path.join(config.DIR_INCEST_MANY, '7_trimmed50.csv')
embeddings_7mer = pd.read_csv(embeddings_7mer_path)
embeddings_7mer

,name,emb_512,emb_128,emb_64,emb_2048,emb_32,emb_1,emb_256,emb_8320,emb_5,...,emb_6176,emb_5376,emb_5888,emb_1109,emb_8193,emb_5525,emb_5477,emb_69,emb_7168,emb_640
0,Gypsy-5_AnMe-I_Gypsy_Anopheles_merus,0.001146,0.000688,0.000000,0.001146,0.000458,0.001146,0.000000,0.000917,0.000917,...,0.000688,0.000688,0.001146,0.000458,0.001146,0.000000,0.000000,0.000688,0.000917,0.000000
1,Gypsy-31_AnMe-I_Gypsy_Anopheles_merus,0.000000,0.000000,0.000226,0.000000,0.000000,0.000000,0.000000,0.000226,0.000226,...,0.000226,0.000000,0.000000,0.000226,0.000000,0.000226,0.000000,0.000000,0.000226,0.000000
2,GYPSY41-LTR_AG_Gypsy_Anopheles_gambiae,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,Gypsy-17_AnSt-I_Gypsy_Anopheles_stephensi,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000526,0.000000,0.000000,0.000000
4,GYPSY55-LTR_AG_Gypsy_Anopheles_gambiae,0.000000,0.000000,0.003984,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102253,DIRS-1F-LTR_DR_DIRS_Danio_rerio,0.000000,0.000000,0.001473,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.001473,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
102254,CR1-44_DR_CR1_Danio_rerio,0.000544,0.000000,0.002176,0.000544,0.000000,0.000544,0.000544,0.000000,0.000000,...,0.000000,0.002720,0.001088,0.001632,0.000544,0.000000,0.000000,0.001088,0.001088,0.000544
102255,DNA-4-5_DR_DNA_transposon_Danio_rerio,0.000000,0.000000,0.001086,0.001086,0.000000,0.002172,0.001086,0.000000,0.000000,...,0.000000,0.000000,0.001086,0.000000,0.000000,0.004343,0.004343,0.000000,0.000000,0.001086
102256,hAT-27_DR_hAT_Danio_rerio,0.000000,0.000000,0.000301,0.000301,0.000000,0.000602,0.000301,0.000602,0.000000,...,0.000903,0.000602,0.000602,0.000602,0.000301,0.000000,0.000000,0.000301,0.000301,0.000602


In [4]:
types_7mer_path = os.path.join(config.DIR_INCEST_MANY, 'repbase_filtered.csv')
types_7mer = pd.read_csv(types_7mer_path, sep=',')
types_7mer

,name,MainType,SubType,Length,Good
0,ISL2EU-51_CGi_ISL2EU_Crassostrea_gigas,DNA transposon,IS-like/prokaryotic,2941,1
1,Kolobok-6_LMi_Kolobok_Locusta_migratoria,DNA transposon,Kolobok,1579,1
2,Transib-2N1_DTa_Transib_Drosophila_takahashii,DNA transposon,Transib,1334,1
3,Tad1-13B_BG_Tad1_Blumeria_graminis,DNA transposon,hAT/Tad1,3824,1
4,MuDR-20_TAe_MuDR_Triticum_aestivum,DNA transposon,MuDR/Mutator,4609,1
...,...,...,...,...,...
24214,L2-46_DR_L2_Danio_rerio,Non-LTR retrotransposon,LINE/L1/L2,2517,1
24215,L1Lx_II_L1_Mus_musculus,Non-LTR retrotransposon,LINE/L1/L2,6088,1
24216,LINE1-N1H_OS_L1_Oryza_sativa,Non-LTR retrotransposon,LINE/L1/L2,1253,1
24217,PteBra-2.35_L1_Pteronura_brasiliensis,Non-LTR retrotransposon,LINE/L1/L2,7031,1


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# merge по name (оставляем только те, у кого есть и эмбеддинг, и MainType)
df = types_7mer.query("Good == 1")[['name', 'MainType']].merge(
    embeddings_7mer,
    on='name',
    how='inner'
)

print("Merged:", df.shape)
print("Unique MainType:", df['MainType'].nunique())
print(df['MainType'].value_counts().head())

Merged: (24517, 52)
Unique MainType: 3
MainType
Non-LTR retrotransposon    8193
LTR retrotransposon        8165
DNA transposon             8159
Name: count, dtype: int64


In [6]:
# признаки: все emb_*
emb_cols = [c for c in df.columns if c.startswith("emb_")]
X = df[emb_cols].to_numpy(dtype=np.float32)

# таргет: MainType -> int
le = LabelEncoder()
y = le.fit_transform(df['MainType'].astype(str))

# train/val split со стратификацией
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "X_val:", X_val.shape)

X_train: (19613, 50) X_val: (4904, 50)


In [ ]:
from scripts.n12_cnn_model import CNNClassifierModel
# модель
input_dim = X_train.shape[1]
class_num = len(le.classes_)

model = CNNClassifierModel(input_dim=input_dim, class_num=class_num)
history = model.train(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=20,
    batch_size=32
)

Epoch 1/20


In [ ]:

# 6инференс + декодирование классов обратно в названия MainType
pred_classes, pred_probs = model.predict(X_val)
pred_labels = le.inverse_transform(pred_classes)

print("Примеры предсказаний:", pred_labels[:10])